## Notebook to learn to play with tif images

In [ ]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import rasterio

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

print(tf.config.list_physical_devices('GPU'))

In [ ]:
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)
# display(settings)

In [ ]:
imp.reload(build_data)

data_generator = build_data.data_generator(settings)

sample_years, sample_lats, sample_lons = build_data.make_sample_list(settings)
print(sample_years.shape, sample_lons.shape, sample_lats.shape)

x_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))
y_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))

# CHECK INTO THIS, NOT SURE HOW THE SHUFFLING WORKS HERE WITH THE TRAIN / VAL SPLIT
# turned off "reshuffle_each_iteration" due to issues with train/val split
x_tfds = x_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=False, seed = settings["rng_seed"])
y_tfds = y_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=False, seed = settings["rng_seed"])

x_tfds = x_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_x_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

y_tfds = y_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_y_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

init_batch_x = next(x_tfds.as_numpy_iterator())
tfds_all = tf.data.Dataset.zip((x_tfds, y_tfds))
tfds_all = tfds_all.prefetch(tf.data.AUTOTUNE)

In [ ]:
imp.reload(build_model)
imp.reload(train_model)

# need to look into whether this is actually doing what we think it is given the BATCH/SHUFFLE above
# also, techincally the way I have this set up there is a non-zero possibility data in the training set is also in the validation set
# I have confirmed that this is NOT correct with the shuffle above.
tfds_train = tfds_all.take(settings["n_batches"][0])
tfds_val = tfds_all.skip(settings["n_batches"][0]).take(settings["n_batches"][1])

model = build_model.build_model(settings, input_shape=np.shape(init_batch_x)[1:])
model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train, tfds_val)